# Adidas LAM Chatbot Analytics

This notebook reproduces the core business-case analysis from the supplied Excel workbook. The workbook supports volume, channel, category, subcategory, classification-failure, and prioritization analysis. The containment, repeat-contact, and resolution KPIs are treated as business-case inputs because the workbook does not contain session IDs, customer IDs, timestamps, CSAT, or resolution outcome labels.

# Section 1 — Imports

In [66]:
from pathlib import Path
import pandas as pd
import numpy as np



# Section 2 — Configuration

In [67]:
DATA_PATH = Path(r"C:\Users\restr\Desktop\adidas-chatbot-case\data\raw\Business_Case_Chatbot_data_Raw_Data.xlsx")

if DATA_PATH.exists():
    print(f" Success! File found at: {DATA_PATH}")
else:
    print(f" ERROR: File NOT found at: {DATA_PATH}")
    print("Please double-check the folder path or spelling.")

 Success! File found at: C:\Users\restr\Desktop\adidas-chatbot-case\data\raw\Business_Case_Chatbot_data_Raw_Data.xlsx


# Section 3 — Cargar Dataset

In [68]:
# Abrir el archivo de Excel para inspeccionar la metadata
xls = pd.ExcelFile(DATA_PATH)

# Mostrar los nombres de las hojas del archivo
print("   Hojas disponibles en el archivo de excel:")
for index, name in enumerate(xls.sheet_names, start=1):
    print(f"  {index}. {name}")

   Hojas disponibles en el archivo de excel:
  1. Agent handled only volume
  2. Hybrid Handled only volume
  3. Bot only volume


# Section 4 — Cargar cada hoja

1. Agent handled only volume --> df_agent
2. Hybrid Handled only volume --> df_hybrid
3. Bot only volume --> df_bot

In [69]:
# Seleccionamos la primera hoja dinámicamente de los metadatos.
df_agent = pd.read_excel(DATA_PATH, sheet_name="Agent handled only volume")
df_hybrid = pd.read_excel(DATA_PATH, sheet_name="Hybrid Handled only volume")
df_bot = pd.read_excel(DATA_PATH, sheet_name="Bot only volume")

Lo anterior lo segmentamos por:

- modelo de gestión
- ruta de escalamiento
- nivel de automatización

Esto significa que Adidas realiza un seguimiento operativo de las conversaciones según el canal de resolución de problemas.


Lo que probablemente representa cada hoja

| Hoja | Significado |
| :--- | :--- |
| Agent handled only volume | Interacciones solo con humanos |
| Hybrid Handled only volume  | Bot + escalamiento humano |
| Bot only volume | Interacciones totalmente automatizadas |

## Por qué esto es extremadamente importante

Esta estructura nos permite analizar:

### A. Eficacia de la automatización
Podemos comparar:
- Lo que el bot resuelve **por sí solo**.
- Lo que requiere **escalamiento**.
- Lo que **evita** la automatización por completo.

### B. Complejidad de la intención/petición
Algunas intenciones/peticiones son:
- Fáciles de automatizar.
- Parcialmente automatizables.
- Imposibles de automatizar de forma segura.

Esta segmentación ayuda a identificarlas.

### C. Fugas de escalamiento
Los flujos híbridos son especialmente importantes porque suelen indicar:
1. Que el bot gestionó parcialmente la solicitud,
2. pero no la resolvió por completo.

> **Nota:** Este es uno de los mayores costes operativos en los sistemas de IA conversacional.

## Perspectiva/Insight estratégica

En última instancia, todo se reduce a:

> **“¿Cómo reducimos las escaladas híbridas innecesarias?”**

Ese es probablemente el verdadero objetivo empresarial.

**Un Insight muy importante.**

# Section 5 — Inspect Shapes

## Agent handled only volume

In [70]:
df_agent.shape

(184, 8)

In [87]:
df_agent.head()

,contact_reason,conteo_de_filas,percentage,unnamed_3,contact_reason1,sub_category,conteo_de_filas1,percentage1
0,Returns & Refunds,116308.0,0.292082,NaN,Customer Feedback,Left Blank,70731,0.177625
1,Existing Order,106339.0,0.267047,NaN,Support on Ordering,Explain how to order,52148,0.130958
2,Customer Feedback,71907.0,0.180579,NaN,Returns & Refunds,Left Blank,36494,0.091647
3,Support on Ordering,52426.0,0.131656,NaN,Returns & Refunds,Return status,30817,0.077390
4,Spam/No Contact,16204.0,0.040693,NaN,Existing Order,Size,20702,0.051989


In [72]:
df_agent.info()

<class 'pandas.DataFrame'>
RangeIndex: 184 entries, 0 to 183
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Contact Reason     16 non-null     str    
 1   Conteo de Filas    18 non-null     float64
 2   Percentage         18 non-null     float64
 3   Unnamed: 3         0 non-null      float64
 4   Contact Reason.1   184 non-null    str    
 5   Sub Category       184 non-null    str    
 6   Conteo de Filas.1  184 non-null    int64  
 7   Percentage.1       184 non-null    float64
dtypes: float64(4), int64(1), str(3)
memory usage: 11.6 KB


### Interpretación

Esto significa:
*   **184** filas
*   **8** columnas

**Pero es importante destacar que:**
El conjunto de datos **NO** contiene datos brutos a nivel de conversación. Se trata de **datos agregados de análisis operacional**.

> Esto modifica significativamente nuestra estrategia analítica.

---

### Consecuencia importante
**NO estamos realizando:**
- Entrenamiento en PLN
- Incrustaciones de conversaciones
- Análisis de transcripciones
- Ajuste de modelos de lenguaje natural (LLM)

**En cambio, estamos realizando:**
- Análisis operacional
- Diagnóstico de KPI
- Análisis de oportunidades de automatización




## Hybrid Handled only volume

In [73]:
df_hybrid.shape

(208, 8)

In [74]:
df_hybrid.head()

,Contact Reason,Conteo de Filas,Percentage,Unnamed: 3,Contact Reason.1,Sub Category,Conteo de Filas.1,Percentage.1
0,Existing Order,113898.0,0.411416,NaN,Spam/No Contact,Left Blank,26688,0.096401
1,Returns & Refunds,74494.0,0.269083,NaN,Existing Order,Size,20571,0.074305
2,Spam/No Contact,26788.0,0.096762,NaN,Returns & Refunds,Return status,18098,0.065373
3,Payment,13149.0,0.047496,NaN,Existing Order,In transit,14978,0.054103
4,Product Information,11036.0,0.039864,NaN,Returns & Refunds,label Request,14028,0.050671


In [75]:
df_hybrid.info()

<class 'pandas.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Contact Reason     19 non-null     str    
 1   Conteo de Filas    19 non-null     float64
 2   Percentage         19 non-null     float64
 3   Unnamed: 3         0 non-null      float64
 4   Contact Reason.1   208 non-null    str    
 5   Sub Category       208 non-null    str    
 6   Conteo de Filas.1  208 non-null    int64  
 7   Percentage.1       208 non-null    float64
dtypes: float64(4), int64(1), str(3)
memory usage: 13.1 KB


### Interpretación
**(208, 8)**

Más filas que con agentes únicamente.

**Esto probablemente sugiere:**
*   Mayor diversidad de intenciones.
*   **O** una taxonomía de escalamiento más fragmentada.

> **Esto resulta interesante desde el punto de vista operativo.**


## Bot only volume

In [76]:
df_bot.shape

(77, 8)

In [77]:
df_bot.head()

,Contact Reason,Conteo de Filas,Percentage,Unnamed: 3,Contact Reason.1,Sub Category,Conteo de Filas.1,Percentage.1
0,Returns & Refunds,54282.0,0.439281,NaN,Returns & Refunds,How to return,19915,0.161164
1,Existing Order,50294.0,0.407008,NaN,Existing Order,Size,19235,0.155661
2,Payment,6618.0,0.053557,NaN,Returns & Refunds,Not defined by Bot,17245,0.139557
3,Not defined by Bot,4833.0,0.039111,NaN,Returns & Refunds,Return status,12934,0.104669
4,Apps & Website,2071.0,0.016760,NaN,Existing Order,Not defined by Bot,10052,0.081347


In [78]:
df_bot.info()

<class 'pandas.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Contact Reason     15 non-null     str    
 1   Conteo de Filas    15 non-null     float64
 2   Percentage         15 non-null     float64
 3   Unnamed: 3         0 non-null      float64
 4   Contact Reason.1   77 non-null     str    
 5   Sub Category       77 non-null     str    
 6   Conteo de Filas.1  77 non-null     int64  
 7   Percentage.1       77 non-null     float64
dtypes: float64(4), int64(1), str(3)
memory usage: 4.9 KB


### Interpretación
**(77, 8)**

Muy pequeño.

**Esto sugiere firmemente que:**
El bot totalmente automatizado gestiona con éxito un **conjunto relativamente limitado de intenciones**.

> **Esto ya representa una importante información para el negocio.**

# Section 6 — Missing Values

In [80]:
df_agent.isna().sum()

Contact Reason       168
Conteo de Filas      166
Percentage           166
Unnamed: 3           184
Contact Reason.1       0
Sub Category           0
Conteo de Filas.1      0
Percentage.1           0
dtype: int64

In [79]:
df_hybrid.isna().sum()

Contact Reason       189
Conteo de Filas      189
Percentage           189
Unnamed: 3           208
Contact Reason.1       0
Sub Category           0
Conteo de Filas.1      0
Percentage.1           0
dtype: int64

In [81]:
df_bot.isna().sum()

Contact Reason       62
Conteo de Filas      62
Percentage           62
Unnamed: 3           77
Contact Reason.1      0
Sub Category          0
Conteo de Filas.1     0
Percentage.1          0
dtype: int64

# Section 7 — Standardize Column Names

In [82]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^\w]", "", regex=True)
    )
    return df

In [83]:
df_agent_clean = clean_columns(df_agent)

In [84]:
df_hybrid_clean = clean_columns(df_hybrid)

In [85]:
df_bot_clean = clean_columns(df_bot)